In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
print(sys.executable)   # should end in sec-10k-rag/.venv/bin/python

/home/nasim/Desktop/work/rag/sec-10k-rag/.venv/bin/python


In [6]:
from pathlib import Path
from sec_10k_rag.parsing.html_text import html_to_text
from sec_10k_rag.parsing.sections import split_into_items

In [7]:
html: bytes = Path("../data/raw/TSLA/2025/10k.html").read_bytes()
text: str = html_to_text(html)
print(text[:2000])

UNITED STATES
SECURITIES AND EXCHANGE COMMISSION
Washington, D.C. 20549
FORM 10-K
(Mark One)
x | ANNUAL REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934
For the fiscal year ended December 31, 2025
OR
o | TRANSITION REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934
For the transition period from _________ to _________
Commission File Number: 001-34756
Tesla, Inc.
(Exact name of registrant as specified in its charter)
Texas | 91-2197729
(State or other jurisdiction of
incorporation or organization) | (I.R.S. Employer
Identification No.)
1 Tesla Road
Austin, Texas
78725
(Address of principal executive offices) | (Zip Code)
(512) 516-8177
(Registrant’s telephone number, including area code)
Securities registered pursuant to Section 12(b) of the Act:
Title of each class | Trading Symbol(s) | Name of each exchange on which registered
Common stock | TSLA | The Nasdaq Global Select Market
Securities registered pursuant to Section 12(g) o

In [8]:
print(text[2000:6000])

her the registrant is a large accelerated filer, an accelerated filer, a non-accelerated filer, a smaller reporting company, or an emerging growth company. See the definitions of “large accelerated filer,” “accelerated filer,” “smaller reporting company” and “emerging growth company” in Rule 12b-2 of the Exchange Act:
Large accelerated filer | x | Accelerated filer | o
Non-accelerated filer | o | Smaller reporting company | o
Emerging growth company | o
If an emerging growth company, indicate by check mark if the registrant has elected not to use the extended transition period for complying with any new or revised financial accounting standards provided pursuant to Section 13(a) of the Exchange Act. o
Indicate by check mark whether the Registrant has filed a report on and attestation to its management’s assessment of the effectiveness of its internal control over financial reporting under Section 404(b) of the Sarbanes-Oxley Act (15 U.S.C. 7262(b)) by the registered public accounting f

In [17]:
from sec_10k_rag.models import Heading, RawSection

from sec_10k_rag.parsing.sections import (
    body_start, find_headings, pick_body_headings, split_into_items,
)


In [10]:
print(f"{len(text):,} characters")

398,871 characters


In [11]:
headings: list[Heading] = find_headings(text)
print(len(headings), "heading-like lines")


46 heading-like lines


In [12]:
print(headings[:10])

[Heading(item='1', start=4865), Heading(item='1A', start=4882), Heading(item='1B', start=4904), Heading(item='1C', start=4939), Heading(item='2', start=4962), Heading(item='3', start=4981), Heading(item='4', start=5007), Heading(item='5', start=5048), Heading(item='6', start=5165), Heading(item='7', start=5184)]


In [13]:
for h in headings:
    line: str = text[h.start : text.find("\n", h.start)]
    print(f"{h.start:>8,}  {h.item:<3} {line}")

   4,865  1   Item 1.
   4,882  1A  Item 1A.
   4,904  1B  Item 1B.
   4,939  1C  Item 1C.
   4,962  2   Item 2.
   4,981  3   Item 3.
   5,007  4   Item 4.
   5,048  5   Item 5.
   5,165  6   Item 6.
   5,184  7   Item 7.
   5,278  7A  Item 7A.
   5,346  8   Item 8.
   5,398  9   Item 9.
   5,491  9A  Item 9A.
   5,524  9B  Item 9B.
   5,551  9C  Item 9C.
   5,638  10  Item 10.
   5,702  11  Item 11.
   5,734  12  Item 12.
   5,838  13  Item 13.
   5,921  14  Item 14.
   5,978  15  Item 15.
   6,030  16  Item 16.
   8,208  1   ITEM 1. BUSINESS
  53,458  1A  ITEM 1A. RISK FACTORS
 136,876  1B  ITEM 1B. UNRESOLVED STAFF COMMENTS
 136,917  1C  ITEM 1C. CYBERSECURITY
 142,644  2   ITEM 2. PROPERTIES
 143,872  3   ITEM 3. LEGAL PROCEEDINGS
 144,099  4   ITEM 4. MINE SAFETY DISCLOSURES
 144,155  5   ITEM 5. MARKET FOR REGISTRANT’S COMMON EQUITY, RELATED STOCKHOLDER MATTERS AND ISSUER PURCHASES OF EQUITY SECURITIES
 146,803  6   ITEM 6. [RESERVED]
 146,822  7   ITEM 7. MANAGEMENT’S DISCUSSIO

In [15]:
start: int = body_start(headings)
print("body starts at heading #", start, "->", headings[start])

body starts at heading # 23 -> Heading(item='1', start=8208)


In [16]:
chosen: list[Heading] = pick_body_headings(headings[start:])
print([h.item for h in chosen])

['1', '1A', '1B', '1C', '2', '3', '4', '5', '6', '7', '7A', '8', '9', '9A', '9B', '9C', '10', '11', '12', '13', '14', '15', '16']


In [18]:
sections: list[RawSection] = split_into_items(text)
for s in sections:
    print(f"{s.item:<3} {len(s.text):>9,}  {s.title:<45} {s.text[:50]!r}")

1      45,249  Business                                      'ITEM 1. BUSINESS\nOverview\nWe are focused on bringi'
1A     83,417  Risk Factors                                  'ITEM 1A. RISK FACTORS\nYou should carefully conside'
1B         40  Unresolved Staff Comments                     'ITEM 1B. UNRESOLVED STAFF COMMENTS\nNone.'
1C      5,726  Cybersecurity                                 'ITEM 1C. CYBERSECURITY\nCybersecurity Risk Manageme'
2       1,227  Properties                                    'ITEM 2. PROPERTIES\nWe are headquartered in Austin,'
3         226  Legal Proceedings                             'ITEM 3. LEGAL PROCEEDINGS\nFor a description of our'
4          55  Mine Safety Disclosures                       'ITEM 4. MINE SAFETY DISCLOSURES\nNot applicable.\nPA'
5       2,647  Market for Registrant's Common Equity         'ITEM 5. MARKET FOR REGISTRANT’S COMMON EQUITY, REL'
6          18  [Reserved]                                    'ITEM 6. [RESERVED]'
7      

In [19]:
from sec_10k_rag.parsing.sgml import split_documents, extract_10k_html

      # the start of the 10-K's HTML

In [20]:
sub: str = Path("../data/raw/TSLA/2025/full_submission.txt").read_text(encoding="utf-8", errors="replace")
docs = split_documents(sub)

In [21]:
print(len(docs), "documents")
print([d.doc_type for d in docs])          # 10-K, EX-..., GRAPHIC, XML, ...
print(extract_10k_html(sub)[:500])  

118 documents
['10-K', 'EX-4.17', 'EX-10.9', 'EX-10.10', 'EX-21.1', 'EX-23.1', 'EX-31.1', 'EX-31.2', 'EX-32.1', 'EX-101.SCH', 'EX-101.CAL', 'EX-101.DEF', 'EX-101.LAB', 'EX-101.PRE', 'GRAPHIC', 'XML', 'XML', 'XML', 'XML', 'XML', 'XML', 'XML', 'XML', 'XML', 'XML', 'XML', 'XML', 'XML', 'XML', 'XML', 'XML', 'XML', 'XML', 'XML', 'XML', 'XML', 'XML', 'XML', 'XML', 'XML', 'XML', 'XML', 'XML', 'XML', 'XML', 'XML', 'XML', 'XML', 'XML', 'XML', 'XML', 'XML', 'XML', 'XML', 'XML', 'XML', 'XML', 'XML', 'XML', 'XML', 'XML', 'XML', 'XML', 'XML', 'XML', 'XML', 'XML', 'XML', 'XML', 'XML', 'XML', 'XML', 'XML', 'XML', 'XML', 'XML', 'XML', 'XML', 'XML', 'XML', 'XML', 'XML', 'XML', 'XML', 'XML', 'XML', 'XML', 'XML', 'XML', 'XML', 'XML', 'XML', 'XML', 'XML', 'XML', 'XML', 'XML', 'XML', 'XML', 'XML', 'XML', 'XML', 'XML', 'XML', 'XML', 'XML', 'XML', 'XML', 'XML', 'XML', 'XML', 'XML', 'XML', 'XML', 'XML', 'JSON', 'ZIP', 'XML']


<?xml version='1.0' encoding='ASCII'?>
<!--XBRL Document Created with the Workiva P